In [ ]:
import torch
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_recall_fscore_support, confusion_matrix

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
model_name = "sentence-transformers/paraphrase-MiniLM-L3-v2"
model = SentenceTransformer(model_name, device=device)
print(f"Loaded model: {model_name}")
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")

In [ ]:
dataset = load_dataset("glue", "mrpc", split="validation")
print("Dataset split: glue/mrpc validation")
print(f"Number of examples in full validation split: {len(dataset)}")
print("Example row:")
print(dataset[0])

In [ ]:
positive_indices = [i for i, label in enumerate(dataset["label"]) if label == 1]
negative_indices = [i for i, label in enumerate(dataset["label"]) if label == 0]

balanced_count_per_class = min(len(positive_indices), len(negative_indices))
selected_indices = negative_indices[:balanced_count_per_class] + positive_indices[:balanced_count_per_class]
balanced_dataset = dataset.select(selected_indices)

labels = balanced_dataset["label"]
sentence1_list = balanced_dataset["sentence1"]
sentence2_list = balanced_dataset["sentence2"]

num_positive = sum(labels)
num_negative = len(labels) - num_positive

print("Constructed balanced subset from validation split")
print(f"Examples per class: {balanced_count_per_class}")
print(f"Balanced subset size: {len(balanced_dataset)}")
print(f"Negative examples: {num_negative}")
print(f"Positive examples: {num_positive}")

In [ ]:
emb1 = model.encode(
    sentence1_list,
    batch_size=64,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=False
)

emb2 = model.encode(
    sentence2_list,
    batch_size=64,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=False
)

cosine_similarities = torch.sum(emb1 * emb2, dim=1).detach().cpu().tolist()

threshold = 0.80
predictions = [1 if score >= threshold else 0 for score in cosine_similarities]
margins = [abs(score - threshold) for score in cosine_similarities]

print(f"Completed embedding inference for {len(predictions)} balanced examples.")
print(f"Fixed cosine similarity threshold: {threshold}")

In [ ]:
accuracy = accuracy_score(labels, predictions)
balanced_accuracy = balanced_accuracy_score(labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary", zero_division=0)
cm = confusion_matrix(labels, predictions)

positive_scores = [score for score, label in zip(cosine_similarities, labels) if label == 1]
negative_scores = [score for score, label in zip(cosine_similarities, labels) if label == 0]
mean_positive_similarity = sum(positive_scores) / len(positive_scores)
mean_negative_similarity = sum(negative_scores) / len(negative_scores)

print("Evaluation metrics:")
print(f"Accuracy         : {accuracy:.4f}")
print(f"Balanced Accuracy: {balanced_accuracy:.4f}")
print(f"Precision        : {precision:.4f}")
print(f"Recall           : {recall:.4f}")
print(f"F1               : {f1:.4f}")
print("Confusion matrix:")
print(cm)
print(f"Mean cosine similarity | label=1: {mean_positive_similarity:.4f}")
print(f"Mean cosine similarity | label=0: {mean_negative_similarity:.4f}")

In [ ]:
label_map = {0: "not_paraphrase", 1: "paraphrase"}
error_indices = [i for i, (y_true, y_pred) in enumerate(zip(labels, predictions)) if y_true != y_pred]
borderline_error_indices = sorted(error_indices, key=lambda i: margins[i])
num_examples_to_show = min(8, len(borderline_error_indices))

print(f"Number of misclassified examples: {len(error_indices)}")
print(f"Showing up to {num_examples_to_show} borderline errors sorted by smallest margin")

for rank, i in enumerate(borderline_error_indices[:num_examples_to_show], start=1):
    row = balanced_dataset[i]
    true_label = labels[i]
    pred_label = predictions[i]
    score = cosine_similarities[i]
    margin = margins[i]
    print(f"Borderline error {rank}")
    print(f"balanced_subset_index: {i}")
    print(f"sentence1: {row['sentence1']}")
    print(f"sentence2: {row['sentence2']}")
    print(f"true label: {true_label} ({label_map[true_label]})")
    print(f"pred label: {pred_label} ({label_map[pred_label]})")
    print(f"cosine similarity: {score:.4f}")
    print(f"distance from threshold: {margin:.4f}")
    print("-" * 80)

In [ ]:
print("RESULT SUMMARY")
print(f"model={model_name}")
print("inference_method=SentenceTransformer_encode_separate_sentence_embeddings_with_cosine_similarity")
print("dataset_split=glue/mrpc validation")
print("subset_type=balanced_equal_positive_negative")
print(f"device={device}")
print(f"num_examples={len(balanced_dataset)}")
print(f"num_negative={num_negative}")
print(f"num_positive={num_positive}")
print(f"threshold={threshold}")
print(f"accuracy={accuracy:.4f}")
print(f"balanced_accuracy={balanced_accuracy:.4f}")
print(f"precision={precision:.4f}")
print(f"recall={recall:.4f}")
print(f"f1={f1:.4f}")
print(f"mean_positive_similarity={mean_positive_similarity:.4f}")
print(f"mean_negative_similarity={mean_negative_similarity:.4f}")